# Phillips Curve Indonesia: Regresi OLS Inflasi vs Pengangguran

**Pertanyaan penelitian:** Apakah terdapat trade-off antara inflasi dan pengangguran di Indonesia (2000–2022)?

**Teori:** Phillips Curve menyatakan hubungan negatif antara inflasi dan pengangguran — saat pengangguran turun, inflasi cenderung naik, dan sebaliknya.

**Metode:** Ordinary Least Squares (OLS) regression

**Sumber data:** World Bank Open Data

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)

print('Setup selesai!')

## 1. Load data

In [ ]:
data = {
    'tahun': list(range(2000, 2023)),
    'inflasi': [
        3.72, 11.50, 11.89, 6.75, 6.06, 10.45, 13.11, 6.41, 9.78, 4.81,
        5.13, 5.36, 4.28, 6.97, 6.42, 6.36, 3.53, 3.81, 3.20, 2.72,
        1.68, 1.56, 4.21
    ],
    'pengangguran': [
        6.08, 8.10, 9.06, 9.67, 9.86, 11.24, 10.28, 9.11, 8.39, 7.87,
        7.14, 6.56, 6.13, 6.17, 5.94, 6.18, 5.61, 5.50, 5.34, 5.28,
        7.07, 6.49, 5.86
    ],
    'pertumbuhan_gdp': [
        4.92, 3.64, 4.50, 4.78, 5.03, 5.69, 5.50, 6.35, 6.01, 4.63,
        6.22, 6.49, 6.23, 5.56, 5.01, 4.79, 5.03, 5.07, 5.17, 5.02,
        -2.07, 3.69, 5.31
    ]
}

df = pd.DataFrame(data).set_index('tahun')
print(f'Data loaded: {df.shape[0]} observasi')
df.head()

## 2. Visualisasi awal — scatter plot Phillips Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Scatter plot
scatter = ax.scatter(
    df['pengangguran'], 
    df['inflasi'],
    c=df.index,          # warna berdasarkan tahun
    cmap='RdYlGn_r',
    s=80, zorder=5, edgecolors='white', linewidth=0.5
)

# Label tiap titik
for tahun, row in df.iterrows():
    ax.annotate(
        str(tahun),
        (row['pengangguran'], row['inflasi']),
        textcoords='offset points', xytext=(6, 4),
        fontsize=7.5, color='#444'
    )

# Trend line
z = np.polyfit(df['pengangguran'], df['inflasi'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['pengangguran'].min(), df['pengangguran'].max(), 100)
ax.plot(x_line, p(x_line), 'steelblue', linestyle='--', linewidth=1.5, 
        label=f'Trend: y = {z[0]:.2f}x + {z[1]:.2f}', zorder=4)

plt.colorbar(scatter, ax=ax, label='Tahun')
ax.set_xlabel('Tingkat Pengangguran (%)', fontsize=12)
ax.set_ylabel('Tingkat Inflasi (%)', fontsize=12)
ax.set_title('Phillips Curve Indonesia 2000–2022\nInflasi vs Pengangguran', 
             fontsize=14, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('../outputs/figures/02_scatter_phillips_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot disimpan!')

## 3. Regresi OLS — Model 1 (simple)

**Model:** $Inflasi_t = \beta_0 + \beta_1 \cdot Pengangguran_t + \varepsilon_t$

In [ ]:
# Variabel dependen dan independen
Y = df['inflasi']
X = sm.add_constant(df['pengangguran'])   # tambah konstanta (intercept)

# Regresi OLS
model1 = sm.OLS(Y, X).fit()

# Tampilkan hasil
print(model1.summary())

## 4. Interpretasi Model 1

> Baca output summary di atas, lalu isi tabel ini:

In [ ]:
# Ringkasan hasil otomatis
beta0 = model1.params['const']
beta1 = model1.params['pengangguran']
r2    = model1.rsquared
pval  = model1.pvalues['pengangguran']

print('=== HASIL REGRESI MODEL 1 ===')
print(f'Intercept (β₀)       : {beta0:.4f}')
print(f'Koefisien (β₁)        : {beta1:.4f}')
print(f'R-squared             : {r2:.4f}')
print(f'P-value pengangguran  : {pval:.4f}')
print()
print('Interpretasi koefisien:')
arah = 'positif' if beta1 > 0 else 'negatif'
print(f'Setiap kenaikan 1% pengangguran → inflasi berubah {beta1:.4f}% ({arah})')
print()
if pval < 0.05:
    print('✓ Koefisien SIGNIFIKAN secara statistik (p < 0.05)')
else:
    print('✗ Koefisien TIDAK signifikan secara statistik (p > 0.05)')
print()
if beta1 < 0:
    print('→ Mendukung teori Phillips Curve (hubungan negatif)')
else:
    print('→ TIDAK mendukung teori Phillips Curve klasik (hubungan positif)')
    print('   Kemungkinan: stagflasi, supply shock, atau struktural ekonomi Indonesia')

## 5. Uji asumsi OLS

Regresi OLS valid hanya jika beberapa asumsi terpenuhi. Kita uji tiga yang paling penting.

In [ ]:
residuals = model1.resid
fitted    = model1.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Uji Asumsi OLS — Model 1', fontsize=14, fontweight='bold')

# Plot 1: Residual vs Fitted (uji homoskedastisitas)
axes[0].scatter(fitted, residuals, color='steelblue', alpha=0.7, edgecolors='white')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residual vs Fitted\n(uji homoskedastisitas)')

# Plot 2: Q-Q plot (uji normalitas residual)
from scipy import stats
(osm, osr), (slope, intercept, r) = stats.probplot(residuals, dist='norm')
axes[1].scatter(osm, osr, color='coral', alpha=0.7, edgecolors='white')
axes[1].plot(osm, slope*np.array(osm)+intercept, 'r--', linewidth=1.5)
axes[1].set_xlabel('Theoretical quantiles')
axes[1].set_ylabel('Sample quantiles')
axes[1].set_title('Q-Q Plot\n(uji normalitas residual)')

# Plot 3: Residual vs Time (uji autokorelasi visual)
axes[2].plot(df.index, residuals, 'o-', color='seagreen', linewidth=1.5, markersize=5)
axes[2].axhline(0, color='red', linestyle='--', linewidth=1)
axes[2].set_xlabel('Tahun')
axes[2].set_ylabel('Residuals')
axes[2].set_title('Residual vs Waktu\n(uji autokorelasi visual)')

plt.tight_layout()
plt.savefig('../outputs/figures/02_uji_asumsi_ols.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Uji statistik formal

# 1. Breusch-Pagan (heteroskedastisitas)
bp_test = het_breuschpagan(residuals, model1.model.exog)
print('=== UJI ASUMSI OLS ===')
print()
print('1. Breusch-Pagan (Heteroskedastisitas)')
print(f'   LM statistic : {bp_test[0]:.4f}')
print(f'   p-value      : {bp_test[1]:.4f}')
if bp_test[1] > 0.05:
    print('   → Tidak ada heteroskedastisitas (asumsi terpenuhi ✓)')
else:
    print('   → Ada heteroskedastisitas (asumsi dilanggar ✗)')

# 2. Durbin-Watson (autokorelasi)
dw = durbin_watson(residuals)
print()
print('2. Durbin-Watson (Autokorelasi)')
print(f'   DW statistic : {dw:.4f}')
print('   (nilai ideal mendekati 2.0)')
if 1.5 < dw < 2.5:
    print('   → Tidak ada autokorelasi signifikan (asumsi terpenuhi ✓)')
else:
    print('   → Ada indikasi autokorelasi (perlu investigasi lebih lanjut ✗)')

# 3. Normalitas (Jarque-Bera)
from statsmodels.stats.stattools import jarque_bera
jb = jarque_bera(residuals)
print()
print('3. Jarque-Bera (Normalitas Residual)')
print(f'   JB statistic : {jb[0]:.4f}')
print(f'   p-value      : {jb[1]:.4f}')
if jb[1] > 0.05:
    print('   → Residual terdistribusi normal (asumsi terpenuhi ✓)')
else:
    print('   → Residual tidak normal (asumsi dilanggar ✗)')

## 6. Model 2 — kontrol dengan pertumbuhan GDP

**Model:** $Inflasi_t = \beta_0 + \beta_1 \cdot Pengangguran_t + \beta_2 \cdot GrowthGDP_t + \varepsilon_t$

Menambahkan variabel kontrol untuk mengisolasi efek murni pengangguran terhadap inflasi.

In [ ]:
X2 = sm.add_constant(df[['pengangguran', 'pertumbuhan_gdp']])
model2 = sm.OLS(Y, X2).fit()
print(model2.summary())

In [ ]:
# Perbandingan Model 1 vs Model 2
print('=== PERBANDINGAN MODEL ===')
print(f'{"":30} {"Model 1":>12} {"Model 2":>12}')
print('-' * 56)
print(f'{"R-squared":30} {model1.rsquared:>12.4f} {model2.rsquared:>12.4f}')
print(f'{"Adj. R-squared":30} {model1.rsquared_adj:>12.4f} {model2.rsquared_adj:>12.4f}')
print(f'{"AIC":30} {model1.aic:>12.4f} {model2.aic:>12.4f}')
print(f'{"BIC":30} {model1.bic:>12.4f} {model2.bic:>12.4f}')
print(f'{"β pengangguran":30} {model1.params["pengangguran"]:>12.4f} {model2.params["pengangguran"]:>12.4f}')
print()
print('Catatan: AIC/BIC yang lebih kecil = model lebih baik')
if model2.rsquared_adj > model1.rsquared_adj:
    print('→ Model 2 lebih baik (adj. R² lebih tinggi dengan variabel kontrol)')
else:
    print('→ Model 1 sudah cukup (variabel kontrol tidak meningkatkan fit)')

## 7. Kesimpulan dan diskusi

### Temuan utama

**Model 1 — Phillips Curve sederhana (Inflasi ~ Pengangguran)**

Hasil regresi OLS menunjukkan koefisien pengangguran sebesar **β₁ = +1.2956** 
(p-value = 0.000), artinya setiap kenaikan 1% tingkat pengangguran justru 
**meningkatkan** inflasi sebesar 1.30%. Hasil ini secara statistik sangat 
signifikan, namun **berlawanan arah dengan prediksi teori Phillips Curve klasik** 
yang menyatakan hubungan negatif antara pengangguran dan inflasi.

R-squared sebesar **0.501** menunjukkan variabel pengangguran mampu menjelaskan 
sekitar 50% variasi inflasi Indonesia periode 2000–2022.

**Model 2 — Dengan variabel kontrol pertumbuhan GDP**

Setelah menambahkan variabel pertumbuhan GDP, koefisien pengangguran sedikit 
berubah menjadi **β₁ = +1.2711** dengan Adj. R-squared meningkat menjadi **0.5085**. 
Model 2 lebih baik berdasarkan Adj. R², meskipun AIC Model 2 (106.02) sedikit 
lebih kecil dari Model 1 (106.55), menunjukkan penambahan variabel kontrol 
memberikan peningkatan fit yang marginal.

---

### Uji asumsi OLS

Ketiga asumsi utama OLS terpenuhi:

| Uji | Statistik | p-value | Hasil |
|-----|-----------|---------|-------|
| Breusch-Pagan (heteroskedastisitas) | 3.5504 | 0.0595 | Terpenuhi ✓ |
| Durbin-Watson (autokorelasi) | 1.6151 | — | Terpenuhi ✓ |
| Jarque-Bera (normalitas residual) | 0.5996 | 0.7410 | Terpenuhi ✓ |

Estimasi OLS bersifat **BLUE** (*Best Linear Unbiased Estimator*) — 
hasil regresi dapat dipercaya secara statistik.

---

### Mengapa hubungannya positif? (bukan negatif seperti teori)

Temuan ini tidak berarti teori Phillips Curve salah — ada beberapa penjelasan 
yang relevan untuk konteks Indonesia:

1. **Supply-side shocks dominan.** Inflasi Indonesia lebih banyak didorong 
   kenaikan harga pangan dan energi (administered prices) daripada tekanan 
   permintaan. Kenaikan harga BBM (2005, 2013) terjadi bersamaan dengan 
   kondisi ekonomi yang belum sepenuhnya pulih — pengangguran masih tinggi 
   tapi inflasi melonjak.

2. **Pasar tenaga kerja informal yang besar.** Sekitar 57% tenaga kerja 
   Indonesia berada di sektor informal (BPS, 2022). Angka pengangguran resmi 
   tidak mencerminkan kondisi pasar tenaga kerja sesungguhnya, sehingga 
   hubungan klasik inflasi-pengangguran tidak terdeteksi dengan baik.

3. **Data agregat nasional.** Phillips Curve secara teori lebih tepat diuji 
   pada level regional atau dengan data frekuensi lebih tinggi (kuartalan/bulanan).

---

### Keterbatasan

- Sampel kecil: hanya 23 observasi (data tahunan 2000–2022)
- Tidak memperhitungkan ekspektasi inflasi (*expectations-augmented Phillips Curve*)
- Variabel pengangguran resmi BPS tidak menangkap pengangguran terselubung
- Periode analisis mencakup berbagai rezim kebijakan yang berbeda

---

### Implikasi kebijakan

Temuan ini mengindikasikan bahwa Bank Indonesia **tidak bisa semata-mata 
mengandalkan kondisi pasar tenaga kerja** sebagai sinyal tekanan inflasi. 
Kebijakan moneter perlu mempertimbangkan faktor sisi penawaran — terutama 
harga komoditas global dan kebijakan harga energi domestik — yang tampaknya 
lebih dominan dalam menentukan inflasi Indonesia.

---

**Referensi:**
- Phillips, A.W. (1958). The Relation between Unemployment and the Rate of 
  Change of Money Wage Rates in the United Kingdom, 1861–1957. *Economica*, 
  25(100), 283–299.
- World Bank. (2024). *World Development Indicators*. https://data.worldbank.org
- BPS. (2022). *Keadaan Ketenagakerjaan Indonesia*. Badan Pusat Statistik.

---
*Analisis oleh: Armandya Danu | Ekonomi Universitas Brawijaya | 29 Mei 2026*